# Nano-U: 4-Fold Leave-One-Scene-Out (LOSO) Training on GPU
**Master's Thesis:** *Extending IoT-Robotics Convergence to Learned Perception*  
**Author:** Tharen Candi (IAS-Lab / VIMP-Lab, University of Padua)

> **Prerequisites:**
> 1. Set runtime to **T4 GPU** (`Runtime` → `Change runtime type` → `T4 GPU`).
> 2. Set your GitHub token in Colab Secrets: Click the **🔑 (Secrets)** tab in the left sidebar, add `GITHUB_TOKEN`, and toggle **Notebook access** ON.
> 3. Ensure `TinyAgri_full.zip` is uploaded to your Google Drive root.

In [ ]:
# 1. Authenticate with GITHUB_TOKEN & Clone Fork
import os
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    if token:
        repo_url = f'https://{token}@github.com/tharencandi/Nano-U.git'
        print('✓ Authenticated via GITHUB_TOKEN from Colab Secrets.')
    else:
        repo_url = 'https://github.com/tharencandi/Nano-U.git'
        print('ℹ GITHUB_TOKEN empty; using public URL.')
except Exception as e:
    token = os.environ.get('GITHUB_TOKEN')
    if token:
        repo_url = f'https://{token}@github.com/tharencandi/Nano-U.git'
        print('✓ Authenticated via GITHUB_TOKEN from environment.')
    else:
        repo_url = 'https://github.com/tharencandi/Nano-U.git'
        print('ℹ GITHUB_TOKEN not found in Secrets. Using public HTTPS URL.')

!rm -rf /content/Nano-U
!git clone {repo_url} /content/Nano-U
%cd /content/Nano-U
!pip install -q tf-keras tensorflow-model-optimization pyyaml

import tensorflow as tf
print('TensorFlow Version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs available:', gpus)
assert len(gpus) > 0, 'ERROR: No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU.'

### 2. Mount Google Drive & Extract Dataset
Extracts `TinyAgri_full.zip` into `/content/Nano-U/data/TinyAgri_full`.

In [ ]:
# 2. Extract Dataset from Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile
zip_candidates = [
    '/content/drive/MyDrive/TinyAgri_full.zip',
    '/content/drive/MyDrive/thesis/TinyAgri_full.zip',
    '/content/TinyAgri_full.zip'
]
zip_path = next((p for p in zip_candidates if os.path.exists(p)), None)
assert zip_path is not None, f'Could not find TinyAgri_full.zip! Checked: {zip_candidates}'

print(f'Extracting {zip_path} into /content/Nano-U/data/...')
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/Nano-U/data/')
print('✓ Extraction complete!')
!ls -la /content/Nano-U/data/TinyAgri_full/

### 3. Generate 4-Fold Leave-One-Scene-Out (LOSO) Splits
Partitions the 1,810 SAM2 frames into zero-leakage splits with a 20-frame temporal buffer gap between training and validation in each scene.

In [ ]:
# 3. Generate 4-Fold Leave-One-Scene-Out (LOSO) Splits
!python scripts/prepare_cross_split.py

### 4. Execute 4-Fold QAD Training Loop
Trains BU-Net teacher (Float32) and Nano-U student (INT8 QAT) across all 4 folds on the GPU.

In [ ]:
# 4. Execute 4-Fold QAD Training Loop on GPU
import os, sys

sep = '=' * 55
for fold_idx in range(4):
    cfg = f'config/TinyAgri_fold{fold_idx}.yaml'
    print(f'\n{sep}\n  STARTING QAD FOR FOLD {fold_idx} ({cfg})\n{sep}')
    exit_code = os.system(f'python -u scripts/run_qad.py --config {cfg}')
    if exit_code != 0:
        raise RuntimeError(f'Fold {fold_idx} failed with exit code {exit_code}!')
    print(f'✓ Fold {fold_idx} finished successfully!')

### 5. Aggregate Summary and Package Deliverables
Saves cross-validation summary statistics, copies the zip archive to Google Drive, and downloads to your browser.

In [ ]:
# 5. Aggregate Summary and Package for Download
import os, json, shutil
from google.colab import files

cv_summary = {}
for i in range(4):
    p = f'results/TinyAgri_fold{i}/pipeline_summary.json'
    if os.path.exists(p):
        with open(p) as f:
            cv_summary[f'fold_{i}'] = json.load(f)

summary_path = 'results/cv_4folds_summary.json'
with open(summary_path, 'w') as f:
    json.dump(cv_summary, f, indent=2)
print(f'✓ Saved cross-validation summary to {summary_path}')

# Print summary table
print('\n' + '-' * 65)
header = f'{"Fold":<8} | {"Test Scene":<18} | {"Teacher mIoU":<14} | {"Student INT8 IoU_fg"}'
print(header)
print('-' * 65)
scene_names = ['Crops_scene1', 'Crops_scene2', 'Tomatoes_scene1', 'Tomatoes_scene2']
for i in range(4):
    f_data = cv_summary.get(f'fold_{i}', {})
    t_miou = f_data.get('teacher', {}).get('eval', {}).get('miou', float('nan'))
    s_eval = f_data.get('student', {}).get('eval_int8', {})
    s_fg = s_eval.get('iou_foreground', s_eval.get('miou', float('nan')))
    row = f'Fold {i:<3} | {scene_names[i]:<18} | {t_miou:<14.4f} | {s_fg:.4f}'
    print(row)
print('-' * 65)

print('\nCompressing models and results into nanou_4folds_trained.zip...')
!zip -r -q nanou_4folds_trained.zip models results config
size_mb = os.path.getsize('nanou_4folds_trained.zip') / 1e6
print(f'✓ Archive ready: nanou_4folds_trained.zip ({size_mb:.2f} MB)')

# Backup to Google Drive root so data is safe even if browser download is blocked
drive_backup = '/content/drive/MyDrive/nanou_4folds_trained.zip'
shutil.copyfile('nanou_4folds_trained.zip', drive_backup)
print(f'✓ Successfully saved backup copy to Google Drive: {drive_backup}')

# Trigger browser download
files.download('nanou_4folds_trained.zip')